# Week 01 — MDPs and Bandits · Workbook

This is a reproducible **reference workbook**. Many supplied scaffold or check
cells directly call `project.solutions`; executing them checks the reference
demonstration, not your implementation. They do not automatically switch modes
when run interactively or with papermill.

For your own attempt, save a copy, replace the relevant reference wrapper with
your code, and make sure the checks call that code. Predict a boundary case
before running it. For independently checked TODO exercises with no fallback,
start with [the progressive coding projects](../docs/INTERACTIVE_LEARNING.md).

**Teaching companion:** [Worked examples, learning bridges and self-checks](teaching_guide.md).


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath('.'))
from common.plotting import use_rl_style, RL_PALETTE
from common.seed import seed_everything
use_rl_style(); seed_everything(0)
import numpy as np
import matplotlib.pyplot as plt
from project import solutions as S
from project.env import BernoulliBandit, SyntheticLeadBrokerBandit
from project.agent import EpsilonGreedyAgent, UCB1Agent, ThompsonBernoulliAgent, LinUCBAgent
print('setup ok')

## Exercise 1 (★) — Incremental sample average
Implement the running mean $Q_n=Q_{n-1}+(R_n-Q_{n-1})/n$ and return the array of estimates after each observation.

> **Hint** (try yourself first): one loop, keep a running `Q` and divide by the **current** count, not the total.

In [ ]:
def exercise_1_incremental_mean(rewards):
    # YOUR CODE HERE
    return S.exercise_1_incremental_mean(rewards)


In [ ]:
rng = np.random.default_rng(0)
r = rng.standard_normal(20)
out = exercise_1_incremental_mean(r)
ref = np.cumsum(r) / np.arange(1, 21)
np.testing.assert_allclose(out, ref, atol=1e-10)
print('Exercise 1: OK')

## Exercise 2 (★) — $\epsilon$-greedy cumulative regret
Run $\epsilon$-greedy with $\epsilon=0.1$ on a 3-arm Bernoulli with means $(0.2, 0.5, 0.8)$ for $T=200$ steps and return the cumulative-regret curve.

> **Hint**: use the `EpsilonGreedyAgent` class and `BernoulliBandit`. The instantaneous regret is `env.regret(a)`.

In [ ]:
def exercise_2_eps_greedy_regret(means, T=200, eps=0.1, seed=0):
    # YOUR CODE HERE
    return S.exercise_2_eps_greedy_regret(means, T=T, eps=eps, seed=seed)


In [ ]:
means = np.array([0.2, 0.5, 0.8])
reg = exercise_2_eps_greedy_regret(means, T=200, eps=0.1, seed=0)
assert reg.shape == (200,)
assert np.all(np.diff(reg) >= -1e-9)
print(f'Exercise 2: final regret = {reg[-1]:.2f}')
fig, ax = plt.subplots(figsize=(5,3))
ax.plot(reg, color=RL_PALETTE['blue'])
ax.set_xlabel('step'); ax.set_ylabel('cumulative regret'); ax.set_title('eps-greedy on 3-arm Bernoulli');
plt.show()

## Exercise 3 (★★) — UCB1 bonus
Compute the Hoeffding bonus $\sqrt{2\log t / N_a}$ for an array of arm-counts. Unplayed arms get $+\infty$.

> **Hint**: `np.where(counts > 0, ..., np.inf)`.

In [ ]:
def exercise_3_ucb1_bonus(counts, t):
    # YOUR CODE HERE
    return S.exercise_3_ucb1_bonus(counts, t)


In [ ]:
b = exercise_3_ucb1_bonus(np.array([1, 10, 100]), t=100)
assert b[0] > b[1] > b[2]
b0 = exercise_3_ucb1_bonus(np.array([0, 5]), t=10)
assert np.isinf(b0[0])
print('Exercise 3: OK', b)

## Exercise 4 (★★) — Thompson posterior update
Given a Beta$(\alpha_0,\beta_0)$ prior and a vector of Bernoulli rewards, return the posterior $(\alpha,\beta)$.

> **Hint**: the conjugate update is $\alpha\leftarrow\alpha+\sum r_i$, $\beta\leftarrow\beta+\sum(1-r_i)$.

In [ ]:
def exercise_4_thompson_posterior(a0, b0, rewards):
    # YOUR CODE HERE
    return S.exercise_4_thompson_posterior(a0, b0, rewards)


In [ ]:
a, b = exercise_4_thompson_posterior(1.0, 1.0, np.array([1, 0, 1, 1, 0]))
assert a == 4.0 and b == 3.0
print('Exercise 4: OK, posterior =', (a, b))

## Exercise 5 (★★) — Bake-off
Compare $\epsilon$-greedy, UCB1, and Thompson on a 4-arm Bernoulli with means $(0.1,0.25,0.5,0.75)$, $T=500$, 5 seeds. Return a dict of final-regret means.

> **Hint**: for each seed, give every agent its own environment instance so reward noise is independent.

In [ ]:
def exercise_5_bakeoff(means, T=500, n_seeds=5):
    # YOUR CODE HERE
    return S.exercise_5_bakeoff(means, T=T, n_seeds=n_seeds)


In [ ]:
means = np.array([0.1, 0.25, 0.5, 0.75])
finals = exercise_5_bakeoff(means, T=500, n_seeds=5)
assert finals['Thompson'] < finals['eps-greedy(0.1)']
for k, v in finals.items():
    print(f'  {k:>20s}: {v:.2f}')

## Exercise 6 (★★) — Discounted return
Compute $G=\sum_{k=0}^{T-1}\gamma^k r_k$. Confirm by hand that $(1,1,1)$ at $\gamma=0.5$ gives $1.75$.

> **Hint**: $\gamma^{0..T-1}\cdot r$, then sum.

In [ ]:
def exercise_6_discounted_return(rewards, gamma):
    # YOUR CODE HERE
    return S.exercise_6_discounted_return(rewards, gamma)


In [ ]:
assert abs(exercise_6_discounted_return(np.array([1.0, 1.0, 1.0]), 0.5) - 1.75) < 1e-12
assert exercise_6_discounted_return(np.array([]), 0.9) == 0.0
print('Exercise 6: OK')

## Exercise 7 (★★, *meaningful objective* — matching) — LinUCB on the synthetic insurance contextual bandit
Run LinUCB vs $\epsilon$-greedy on `SyntheticLeadBrokerBandit(K=6, d=5)` for $T=1500$. Return both final regrets. Compare across seeds and explain the observed difference. The sigmoid reward model is not exactly linear, so there is no per-run guarantee that LinUCB wins.

> **Hint**: at every step, draw `x = env.context()`, ask each agent for its action, observe `env.pull(x, a)`, and update.

In [ ]:
def exercise_7_linucb_regret(K=6, d=5, T=1500, alpha=1.0, seed=0):
    # YOUR CODE HERE
    return S.exercise_7_linucb_regret(K=K, d=d, T=T, alpha=alpha, seed=seed)


In [ ]:
lin_r, eps_r = exercise_7_linucb_regret(K=6, d=5, T=1500, alpha=1.0, seed=0)
print(f'LinUCB final regret = {lin_r:.2f}; eps-greedy = {eps_r:.2f}')
assert np.isfinite([lin_r, eps_r]).all() and min(lin_r, eps_r) >= 0
print('One seeded comparison; repeat across seeds before judging performance.')

## Exercise 8 (★★★, open-ended — safety + matching) — SNIPS off-policy estimator
Implement the Self-Normalised IPS estimator $\widehat V(\pi) = \sum_i w_i r_i / \sum_i w_i$ where $w_i = \pi(a_i\mid x_i)/\pi_b(a_i\mid x_i)$. Use it to estimate the value of a synthetic insurance-style $\epsilon$-greedy candidate policy from logs of a uniform behaviour policy. Then **discuss in a markdown cell at the bottom** when SNIPS is biased vs. when it is dangerous.

> **Hint**: the denominator is unstable when $\pi$ and $\pi_b$ disagree on most rounds. Inspect support, weight concentration, and effective sample size. Clipping can stabilise arithmetic but adds bias and cannot repair missing coverage.

In [ ]:
def exercise_8_snips(rewards, target_prob, logged_prob):
    # YOUR CODE HERE
    return S.exercise_8_snips(rewards, target_prob, logged_prob)


In [ ]:
seed_everything(1)
env = SyntheticLeadBrokerBandit(K=4, d=4, seed=42)
K = env.K
n_logs = 2000
rng = np.random.default_rng(0)
rewards, target_p, logged_p = [], [], []
for _ in range(n_logs):
    x = env.context()
    a = int(rng.integers(0, K))
    r = env.pull(x, a)
    er = env.expected_reward(x)
    pi_a = np.full(K, 0.2 / K); pi_a[er.argmax()] += 0.8
    rewards.append(r); target_p.append(pi_a[a]); logged_p.append(1.0 / K)
v_hat = exercise_8_snips(np.array(rewards), np.array(target_p), np.array(logged_p))
print(f'SNIPS estimate of pi value: {v_hat:.3f}')
n_mc = 5000; total = 0.0
for _ in range(n_mc):
    x = env.context(); er = env.expected_reward(x)
    if rng.random() < 0.2:
        a = int(rng.integers(0, K))
    else:
        a = int(er.argmax())
    total += env.pull(x, a)
v_mc = total / n_mc
print(f'MC ground truth:           {v_mc:.3f}')
assert abs(v_hat - v_mc) < 0.10, (v_hat, v_mc)

**Discussion.** SNIPS is *biased* in finite samples because the self-normalisation denominator is correlated with the numerator; under suitable moment and coverage assumptions, the bias vanishes asymptotically (often at order $1/n$). SNIPS is *dangerous* in three concrete cases: (i) when the target policy puts mass on actions the behaviour policy almost never logged, so a single high-importance-weight row dominates the estimate; (ii) when the propensities $\pi_b(a\mid x)$ are themselves estimated and unbounded near zero — inspect their estimation error and overlap; (iii) when the reward distribution has heavy tails — estimator variance may remain large; alternative estimators still require their own assumptions.

<!-- AUTO-HINTS-SECTION -->
## Hints toward answers

Attempt the exercise first. Before opening a hint, write the quantity you are
estimating, the data source, and the expected update direction. The
[teaching guide](teaching_guide.md) supplies worked examples, assumptions,
and transfer questions. The hints below collect the exercise's own inline
hints; they do not reveal or execute reference solutions.

<details>
<summary>Exercise 1 (★) — Incremental sample average</summary>

> **Hint** (try yourself first): one loop, keep a running `Q` and divide by the **current** count, not the total.

</details>

<details>
<summary>Exercise 2 (★) — $\epsilon$-greedy cumulative regret</summary>

> **Hint**: use the `EpsilonGreedyAgent` class and `BernoulliBandit`. The instantaneous regret is `env.regret(a)`.

</details>

<details>
<summary>Exercise 3 (★★) — UCB1 bonus</summary>

> **Hint**: `np.where(counts > 0, ..., np.inf)`.

</details>

<details>
<summary>Exercise 4 (★★) — Thompson posterior update</summary>

> **Hint**: the conjugate update is $\alpha\leftarrow\alpha+\sum r_i$, $\beta\leftarrow\beta+\sum(1-r_i)$.

</details>

<details>
<summary>Exercise 5 (★★) — Bake-off</summary>

> **Hint**: for each seed, give every agent its own environment instance so reward noise is independent.

</details>

<details>
<summary>Exercise 6 (★★) — Discounted return</summary>

> **Hint**: $\gamma^{0..T-1}\cdot r$, then sum.

</details>

<details>
<summary>Exercise 7 (★★, *meaningful objective* — matching) — LinUCB on the synthetic insurance contextual bandit</summary>

> **Hint**: at every step, draw `x = env.context()`, ask each agent for its action, observe `env.pull(x, a)`, and update.

</details>

<details>
<summary>Exercise 8 (★★★, open-ended — safety + matching) — SNIPS off-policy estimator</summary>

> **Hint**: the denominator is unstable when $\pi$ and $\pi_b$ disagree on most rounds. Inspect support, weight concentration, and effective sample size. Clipping can stabilise arithmetic but adds bias and cannot repair missing coverage.

</details>

After solving, change one input or assumption and predict the result before
running it. If a cell calls `project.solutions`, passing it verifies the
reference demonstration; test your own implementation separately.
